In [ ]:
import os
from pathlib import Path

import pandas as pd


def find_notebook_dir() -> Path:
    """Locate the notebooks/ folder regardless of the kernel's working directory."""
    # VS Code exposes the notebook path directly.
    vsc_path = globals().get("__vsc_ipynb_file__")
    if vsc_path:
        return Path(vsc_path).resolve().parent
    # Otherwise walk up from cwd looking for this notebook.
    for d in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (d, d / "notebooks"):
            if (candidate / "explore_dataset.ipynb").exists():
                return candidate.resolve()
    raise FileNotFoundError("Cannot locate notebooks/ folder; start the kernel inside the project.")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

In [ ]:
data = pd.read_csv("../data/5k_items_curated.csv")
queries = pd.read_csv("../data/queries.csv")

In [ ]:
data.head()

In [ ]:
data.keys()

In [ ]:
import json

IMAGE_URL_PREFIX = "https://static.ifood-static.com.br/image/upload/t_low/pratos/"

def get_image_from_id(item_id):
    """
    Get the image URLs of the item from the item id.
    To access item images, use the following URL prefix with the corresponding image_str from itemMetadata:
    https://static.ifood-static.com.br/image/upload/t_low/pratos/<image_str>
    The image_str values live in the `images` list inside the itemMetadata JSON.
    Returns an empty list if the item has no images.
    """
    rows = data.loc[data['itemId'] == item_id, 'itemMetadata']
    if rows.empty:
        raise KeyError(f"itemId not found: {item_id}")
    metadata = json.loads(rows.iloc[0])
    return [IMAGE_URL_PREFIX + image_str for image_str in metadata.get('images') or []]

In [ ]:
from io import BytesIO
from urllib.request import urlopen

import matplotlib.pyplot as plt
from PIL import Image
import random


def fetch_image(url):
    """Download an image from a URL and return it as a PIL Image."""
    with urlopen(url, timeout=10) as response:
        return Image.open(BytesIO(response.read())).convert("RGB")


def show_item_images(item_id):
    """Fetch and display all images of an item, titled with the item name."""
    urls = get_image_from_id(item_id)
    name = json.loads(data.loc[data['itemId'] == item_id, 'itemMetadata'].iloc[0])['name']
    if not urls:
        print(f"No images for item {item_id} ({name})")
        return

    fig, axes = plt.subplots(1, len(urls), figsize=(4 * len(urls), 4), squeeze=False)
    for ax, url in zip(axes[0], urls):
        ax.imshow(fetch_image(url))
        ax.axis("off")
    fig.suptitle(name)
    plt.show()


show_item_images(data['itemId'].iloc[random.randint(0, len(data) - 1)])